In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Ρυθμίσεις

In [ ]:
TARGET_IDX = 35

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"

methods = ["IG", "IxG", "Saliency", "GradientSHAP", "DeepSHAP", "KernelSHAP",
           "Grad-CAM", "Grad-CAM++", "Ablation-CAM", "Score-CAM", "Eigen-CAM"]

### Φόρτωση αποθηκευμένων attributions

In [ ]:
attributions = {}
for m in methods:
    path = f"{save_dir}/{m}_idx{TARGET_IDX}.npy"
    attributions[m] = np.load(path)
    print(f"{m}: shape={attributions[m].shape}, sum={attributions[m].sum():.4f}")

### Απόσταση attributions από την ομοιόμορφη κατανομή (flatness)

In [ ]:
uniform = np.ones(50) / 50
flatness = {m: jensenshannon(a, uniform, base=2) for m, a in attributions.items()}
flatness = dict(sorted(flatness.items(), key=lambda kv: kv[1]))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(list(flatness.keys()), list(flatness.values()), color="steelblue")
ax.set_xlabel("JSD από την ομοιόμορφη κατανομή")
ax.set_title("Απόσταση JSD από την ομοιόμορφη κατανομή ανά μέθοδο")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Συναρτήσεις μετρικών

In [ ]:
def jaccard_topk(a, b, k=10):
    """
    Δείκτης Jaccard μεταξύ των k segments με τη μεγαλύτερη σημασία σε δύο
    διανύσματα attribution. Τιμή 1 σημαίνει ίδιο top-k σύνολο.
    """
    top_a = set(np.argsort(a)[-k:])
    top_b = set(np.argsort(b)[-k:])
    return len(top_a & top_b) / len(top_a | top_b)


def saliency_centroid(attr):
    """
    Υπολογίσμός της centroid distance μεταξύ δύο attribution διανυσμάτων. Δείχνει
    πού συγκεντρώνεται κατά μέσο όρο η σημασία, ανεξάρτητα από το σχήμα της κατανομής.
    """
    positions = np.arange(len(attr))
    return np.sum(positions * attr) / np.sum(attr)

### Υπολογισμός μετρικών ανά ζεύγος μεθόδων

In [ ]:
results = []
for i, m1 in enumerate(methods):
    for m2 in methods[i+1:]:
        pearson_r, _ = pearsonr(attributions[m1], attributions[m2])
        spearman_r, _ = spearmanr(attributions[m1], attributions[m2])
        jaccard = jaccard_topk(attributions[m1], attributions[m2], k=10)
        jsd = jensenshannon(attributions[m1], attributions[m2], base=2)
        centroid_dist = abs(saliency_centroid(attributions[m1]) - saliency_centroid(attributions[m2]))

        results.append({
            "method_1": m1, "method_2": m2,
            "pearson": pearson_r, "spearman": spearman_r,
            "jaccard_top10": jaccard,
            "jsd": jsd,
            "centroid_distance": centroid_dist
        })

df = pd.DataFrame(results)
df_sorted = df.sort_values("jsd", ascending=False)
print(df_sorted.to_string(index=False))

### Heatmaps μετρικών συσχέτισης μεθόδων

In [ ]:
all_methods = methods
full_matrix = pd.DataFrame(index=all_methods, columns=all_methods, dtype=float)

for m in all_methods:
    full_matrix.loc[m, m] = 1.0

for _, row in df.iterrows():
    full_matrix.loc[row["method_1"], row["method_2"]] = row["pearson"]
    full_matrix.loc[row["method_2"], row["method_1"]] = row["pearson"]

plt.figure(figsize=(12, 10))
sns.heatmap(full_matrix.astype(float), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"label": "Pearson r"})
plt.title(f"Pearson Correlation μεταξύ μεθόδων επεξηγησιμότητας (idx={TARGET_IDX})")
plt.tight_layout()
plt.show()

In [ ]:
# Πλήρης, συμμετρικός πίνακας για JSD
full_jsd = pd.DataFrame(index=methods, columns=methods, dtype=float)
for m in methods:
    full_jsd.loc[m, m] = 0.0
for _, row in df.iterrows():
    full_jsd.loc[row["method_1"], row["method_2"]] = row["jsd"]
    full_jsd.loc[row["method_2"], row["method_1"]] = row["jsd"]

plt.figure(figsize=(12, 10))
sns.heatmap(full_jsd.astype(float), annot=True, fmt=".2f", cmap="viridis_r",
            vmin=0, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"label": "Jensen-Shannon Divergence"})
plt.title(f"Jensen-Shannon Divergence μεταξύ μεθόδων επεξηγησιμότητας (idx={TARGET_IDX})")
plt.tight_layout()
plt.show()

# Πλήρης, συμμετρικός πίνακας για centroid distance
full_centroid = pd.DataFrame(index=methods, columns=methods, dtype=float)
for m in methods:
    full_centroid.loc[m, m] = 0.0
for _, row in df.iterrows():
    full_centroid.loc[row["method_1"], row["method_2"]] = row["centroid_distance"]
    full_centroid.loc[row["method_2"], row["method_1"]] = row["centroid_distance"]

plt.figure(figsize=(12, 10))
sns.heatmap(full_centroid.astype(float), annot=True, fmt=".1f", cmap="magma_r",
            square=True, linewidths=0.5,
            cbar_kws={"label": "Απόσταση κέντρου βάρους (segments)"})
plt.title(f"Saliency Centroid Distance μεταξύ μεθόδων (idx={TARGET_IDX})")
plt.tight_layout()
plt.show()

### Μέση απόλυτη συσχέτιση ανά μέθοδο

In [ ]:
# Μέση απόλυτη συσχέτιση: το πρόσημο αγνοείται. Το Eigen-CAM (r ≈ -0.95 με τις CAM)
# εμφανίζεται εδώ με υψηλή τιμή, ενώ διαφωνεί με αυτές· βλ. και signed mean παρακάτω.
mean_abs_corr = full_matrix.abs().apply(lambda row: (row.sum() - 1) / (len(row) - 1), axis=1)

# Μέση προσημασμένη συσχέτιση
mean_signed_corr = full_matrix.apply(lambda row: (row.sum() - 1) / (len(row) - 1), axis=1)

summary = pd.DataFrame({"mean_abs_corr": mean_abs_corr, "mean_signed_corr": mean_signed_corr})
print(summary.sort_values("mean_abs_corr", ascending=False).round(3))